In [ ]:
from dotenv import load_dotenv
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END, add_messages
from langchain_core.messages import AnyMessage
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from IPython.display import Image, display

load_dotenv()

### Not Strucured output

In [ ]:

class ResearchState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]   # reducer required
    research: str | None                                  # custom field for middleware


@wrap_model_call
def custom_promt(request, handler):
    research = request.state.get("research")

    # make SystemMessage, HumanMessage
    if research:
        messages = [
            {
                "role": "system",
                "content": f"Research Context:\n{research}"
            },
            *request.messages,   # original messages follow after
        ]
        request = request.override(messages=messages)

    return handler(request) # once retunred, ai invoke call with messages value, output append to messages again.


# Node Definition:
#    Here, the agent created via create_agent acts as a graph node.
# Input Modification:
#    Messages can be intercepted and overridden by middleware 'custom_promt' before reaching the LLM.
# Message History: 
#    AI responses are automatically appended to the messages attribute using the add_messages reducer.
# Structured Result (not implemented): 
#    When a response_format is defined in create_agent using 'response_format', 
#    the final structured result is automatically saved (overwritten) into the 'structured_output' attribute of the state.
# Limitation: 
#    If you want the data in a custom-named field like research_summary, you still need a wrapper function to save the data.
researcher_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[],                          # no tools for this demo
    state_schema=ResearchState,        # expose custom fields to the agent
    middleware=[custom_promt],         # intercept & modify messages before LLM
    system_prompt="You are a helpful research assistant."
)


builder = StateGraph(ResearchState)

builder.add_node("researcher", researcher_agent)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:

result = graph.invoke({
    "research": "Transformers are a type of neural network architecture based on self-attention.",
    "messages": [
        {
            "role": "user",
            "content": "Explain this topic in simple terms, in less than 100 words."
        }
    ]
})

# The agent appended all messages (human + AI) back into state['messages'].
# print(result["messages"][-1].content)
result

### Strucured output (using response_format arg)

In [ ]:
from pydantic import BaseModel, Field
from typing import List

@wrap_model_call
def inject_research(request, handler):
    """
    Middleware to dynamically inject context from the State into the LLM prompt.
    This modifies the 'in-flight' request without polluting the State history.
    """
    research = request.state.get("research")
    if research:
        # Prepend research context as a temporary System Message
        messages = [
            {"role": "system", "content": f"Research Context:\n{research}"},
            *request.messages,
        ]
        request = request.override(messages=messages)
    return handler(request)



class ResearchSummary(BaseModel):
    topic: str = Field(description="The main subject of research")
    concepts: List[str] = Field(description="List of 3 core concepts found")
    complexity_level: str = Field(description="Beginner, Intermediate, or Advanced")


class ResearchDataState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    research: str | None

# Limitation: 
#    If you want the structured data in a custom-named field like research_summary in the state
#    so that other nodes can be accessed, you still need a wrapper function and keep llm inside it to save the data.
researcher_agent_structured = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[],
    state_schema=ResearchDataState,
    middleware=[inject_research],
    system_prompt="Analyze the research and extract a structured summary.",
    response_format=ResearchSummary 
)



In [ ]:
builder = StateGraph(ResearchState)

builder.add_node("researcher_structured", researcher_agent_structured)

builder.add_edge(START, "researcher_structured")
builder.add_edge("researcher_structured", END)

graph_st = builder.compile()
display(Image(graph_st.get_graph().draw_mermaid_png()))

In [ ]:

input_state = {
    "research": "Transformers are neural networks that use self-attention to process sequential data.",
    "messages": [
        {"role": "user", "content": "Analyze this research context and summarize it for me."}
    ]
}

result_str = graph_st.invoke(input_state)
result_str

In [ ]:
ai_msg = result_str["messages"][-2] 
structured_data = ai_msg.tool_calls[0]["args"]
print(structured_data["topic"])
print(structured_data["concepts"])

NOTE:

The create_agent() function primarily manages message history and does not automatically map structured outputs into custom state attributes. In a multi-node graph, this means downstream nodes cannot predictably access data generated by a previous agent without searching through the message history. To solve this, we wrap the agent inside a custom Python node. This allows us to manually extract the LLM's response and explicitly populate the state's attribute, ensuring that specific data is cleanly available for the next node

### Wrap the LLM inside node

In [ ]:
# The LLM
researcher_brain = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ResearchSummary, # AI will use tool-calls internally
    state_schema=ResearchState,
    middleware=[inject_research]
)

# Create Node, use LLM inside node (this is normal pattern in LangGraph)
def researcher_node(state: ResearchState):
    # Call the agent
    result = researcher_brain.invoke(state)
    
    # Safely find the tool call (don't use hardcoded index -2)
    ai_msg = next(m for m in reversed(result["messages"]) if m.tool_calls)
    raw_data = ai_msg.tool_calls[0]["args"]
    
    # save it in state.
    return {
        "messages": result["messages"],
        "structured_output": ResearchSummary(**raw_data) # Populates the 'bucket'
    }


builder = StateGraph(ResearchState)
builder.add_node("researcher", researcher_node) # Add the wrapper, not the brain!


``` The create_agent() function primarily manages message history and does not automatically map structured outputs into custom state attributes. In a multi-node graph, this means downstream nodes cannot predictably access data generated by a previous agent without searching through the message history. To solve this, we wrap the agent inside a custom Python node. This allows us to manually extract the LLM's response and explicitly populate the state, ensuring that specific data is cleanly available for the next node.